# Removing all eddies with ice cover in 3 eddy radii (3R)

removing all eddies that have some kind of ice cover in a 3 radii distance around the eddy centre

#### Data Import

In [1]:
import pandas as pd
import intake
import numpy as np
import matplotlib.pylab as plt
import xarray as xr
import matplotlib.colors as colors
from shapely.geometry import Polygon, Point, box
from shapely import contains_xy
import time
import shapely
from shapely.affinity import scale

In [2]:
#importing data
eerie_cat=intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
da_ocean = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["ocean"]["gr025"]["2d_daily_mean"].to_dask()

In [4]:
#import eddy data with ice free centre 

edso_ac_0 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso_ac.nc")
edso_c_0 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso_c.nc")

### create mask for 3R and calculate mean ice cover (conc)

In [5]:
#calculate conc in 3R for ac eddies
#starting timer
start_time = time.time()
    
#select variable conc for ocean
da_ocean_conc = da_ocean["conc"].squeeze()


def dlon(lon, lon0):
    return ((lon - lon0 + 180) % 360) - 180

#creating new data.frame with only the effective_contour_lat/lon and time of the eddies
dt_contour_ac = xr.Dataset(
        {
            "contour_lat": edso_ac_0["effective_contour_latitude"],
            "contour_lon": edso_ac_0["effective_contour_longitude"],
            "time" : edso_ac_0["time"],
            "radius" : edso_ac_0["effective_radius"],
            "lat" : edso_ac_0["latitude"],
            "lon" : edso_ac_0["longitude"],
            "ID" : edso_ac_0["ID"]
        }
    )

n_obs_ac = dt_contour_ac.sizes["obs"]
    
#changing radius in m to lat/lon, using simplification 
# 1° lat = 111.32 km = 111320 m
# 1° lon = 111.32 km * cos(lat of eddy cetnre) = 111320 m * cos(lat of eddy centre)
    
radius_lat = dt_contour_ac["radius"].values / 111320
radius_lon = dt_contour_ac["radius"].values /(111320 * np.cos(np.deg2rad(dt_contour_ac["lat"].values)))
    
#add new radius_lat/lon to dt_contour
dt_contour_ac = dt_contour_ac.assign(
    radius_lat=("obs", radius_lat),
    radius_lon=("obs", radius_lon))

# Ocean grid
lat_vals = da_ocean_conc["lat"].values
lon_vals = da_ocean_conc["lon"].values

# Eddy properties
eddy_lat = dt_contour_ac["lat"].values
eddy_lon = dt_contour_ac["lon"].values
eddy_time = dt_contour_ac["time"].values
radius_lat = dt_contour_ac["radius_lat"].values * 3 
radius_lon = dt_contour_ac["radius_lon"].values * 3

# Results
mean_arr_ac = np.full(n_obs_ac, np.nan)
npoints_arr_ac = np.zeros(n_obs_ac, dtype=int)

# Process eddies grouped by unique time
unique_times = np.unique(eddy_time)

for t in unique_times:
    # Select ocean data for this time only
    da_ocean_time = da_ocean_conc.sel(time=t, method="nearest").values

    # Indices of eddies at this time
    idx_time = np.where(eddy_time == t)[0]

    for i in idx_time:
        lat0, lon0 = eddy_lat[i], eddy_lon[i]
        rad_lat_i, rad_lon_i = radius_lat[i], radius_lon[i]

       # Local window around eddy
        lat_mask = (lat_vals >= lat0 - rad_lat_i) & (lat_vals <= lat0 + rad_lat_i)
        dlon_vals = ((lon_vals - lon0 + 180) % 360) - 180
        lon_mask = np.abs(dlon_vals) <= rad_lon_i
        lat_sub = lat_vals[lat_mask]
        lon_sub = lon_vals[lon_mask]

        lon_grid_sub, lat_grid_sub = np.meshgrid(lon_sub, lat_sub)

        # Ellipse mask
        mask = (dlon(lon_grid_sub, lon0)/rad_lon_i)**2 + ((lat_grid_sub - lat0)/rad_lat_i)**2 <= 1

        var_sub = da_ocean_time[lat_mask, :][:, lon_mask]

       # Apply mask and remove NaNs
        var_masked = var_sub[mask]
        var_masked = var_masked[np.isfinite(var_masked)]  # <- ignores NaNs

        if var_masked.size > 0:
            mean_arr_ac[i] = var_masked.mean()
            npoints_arr_ac[i] = var_masked.size
        else:
            mean_arr_ac[i] = np.nan
            npoints_arr_ac[i] = 0

# Build dataset
dt_masked_ac = xr.Dataset(
    {
        "mean_conc": ("obs", mean_arr_ac),
        "npoints_ac": ("obs", npoints_arr_ac),
        "ID": ("obs", dt_contour_ac["ID"].values),
    },
    coords={"obs": dt_contour_ac["obs"]}
)

        
#stopping timer
total_time = time.time() - start_time
print(f"Completed: {n_obs_ac} obs in {total_time:.1f} seconds")
estimated_time = (total_time / n_obs_ac) * len(dt_contour_ac["obs"])
print(f"Would complete {len(dt_contour_ac['obs'])} obs in {estimated_time / 3600:.2f} hours")

Completed: 1982340 obs in 362.7 seconds
Would complete 1982340 obs in 0.10 hours


In [6]:
dt_masked_ac

<xarray.Dataset>
Dimensions:     (obs: 1982340)
Coordinates:
  * obs         (obs) int64 0 1 2 3 4 ... 1982336 1982337 1982338 1982339
Data variables:
    mean_conc   (obs) float64 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    npoints_ac  (obs) int64 28 27 33 58 71 64 40 65 ... 20 22 23 25 36 35 34 22
    ID          (obs) float64 1.0 2.0 3.0 4.0 ... 1.987e+06 1.987e+06 1.987e+06

In [7]:
#create conc in 3R for c eddies
#starting timer
start_time = time.time()
    
#select variable conc for ocean
da_ocean_conc = da_ocean["conc"].squeeze()

def dlon(lon, lon0):
    return ((lon - lon0 + 180) % 360) - 180

#creating new data.frame with only the effective_contour_lat/lon and time of the eddies
dt_contour_c = xr.Dataset(
        {
            "contour_lat": edso_c_0["effective_contour_latitude"],
            "contour_lon": edso_c_0["effective_contour_longitude"],
            "time" : edso_c_0["time"],
            "radius" : edso_c_0["effective_radius"],
            "lat" : edso_c_0["latitude"],
            "lon" : edso_c_0["longitude"],
            "ID" : edso_c_0["ID"]
        }
    )

n_obs_c = dt_contour_c.sizes["obs"]
    
#changing radius in m to lat/lon, using simplification 
# 1° lat = 111.32 km = 111320 m
# 1° lon = 111.32 km * cos(lat of eddy cetnre) = 111320 m * cos(lat of eddy centre)
    
radius_lat = dt_contour_c["radius"].values / 111320
radius_lon = dt_contour_c["radius"].values /(111320 * np.cos(np.deg2rad(dt_contour_c["lat"].values)))
    
#add new radius_lat/lon to dt_contour
dt_contour_c = dt_contour_c.assign(
    radius_lat=("obs", radius_lat),
    radius_lon=("obs", radius_lon))

# Ocean grid
lat_vals = da_ocean_conc["lat"].values
lon_vals = da_ocean_conc["lon"].values

# Eddy properties
eddy_lat = dt_contour_c["lat"].values
eddy_lon = dt_contour_c["lon"].values
eddy_time = dt_contour_c["time"].values
radius_lat = dt_contour_c["radius_lat"].values * 3
radius_lon = dt_contour_c["radius_lon"].values * 3

# Results
mean_arr_c = np.full(n_obs_c, np.nan)
npoints_arr_c = np.zeros(n_obs_c, dtype=int)

# Process eddies grouped by unique time
unique_times = np.unique(eddy_time)

for t in unique_times:
    # Select ocean data for this time only
    da_ocean_time = da_ocean_conc.sel(time=t, method="nearest").values

    # Indices of eddies at this time
    idx_time = np.where(eddy_time == t)[0]

    for i in idx_time:
        lat0, lon0 = eddy_lat[i], eddy_lon[i]
        rad_lat_i, rad_lon_i = radius_lat[i], radius_lon[i]

       # Local window around eddy
        lat_mask = (lat_vals >= lat0 - rad_lat_i) & (lat_vals <= lat0 + rad_lat_i)
        dlon_vals = ((lon_vals - lon0 + 180) % 360) - 180
        lon_mask = np.abs(dlon_vals) <= rad_lon_i
        lat_sub = lat_vals[lat_mask]
        lon_sub = lon_vals[lon_mask]

        lon_grid_sub, lat_grid_sub = np.meshgrid(lon_sub, lat_sub)

        # Ellipse mask
        mask = (dlon(lon_grid_sub, lon0)/rad_lon_i)**2 + ((lat_grid_sub - lat0)/rad_lat_i)**2 <= 1

        var_sub = da_ocean_time[lat_mask, :][:, lon_mask]

       # Apply mask and remove NaNs
        var_masked = var_sub[mask]
        var_masked = var_masked[np.isfinite(var_masked)]  # <- ignores NaNs

        if var_masked.size > 0:
            mean_arr_c[i] = var_masked.mean()
            npoints_arr_c[i] = var_masked.size
        else:
            mean_arr_c[i] = np.nan


# Build dataset
dt_masked_c = xr.Dataset(
    {
        "mean_conc": ("obs", mean_arr_c),
        "npoints_ac": ("obs", npoints_arr_c),
        "ID": ("obs", dt_contour_c["ID"].values),
    },
    coords={"obs": dt_contour_c["obs"]}
)

        
#stopping timer
total_time = time.time() - start_time
print(f"Completed: {n_obs_c} obs in {total_time:.1f} seconds")
estimated_time = (total_time / n_obs_c) * len(dt_contour_c["obs"])
print(f"Would complete {len(dt_contour_c['obs'])} obs in {estimated_time / 3600:.2f} hours")

Completed: 1945338 obs in 280.1 seconds
Would complete 1945338 obs in 0.08 hours


In [8]:
dt_masked_c

<xarray.Dataset>
Dimensions:     (obs: 1945338)
Coordinates:
  * obs         (obs) int64 0 1 2 3 4 ... 1945334 1945335 1945336 1945337
Data variables:
    mean_conc   (obs) float64 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    npoints_ac  (obs) int64 173 175 154 150 150 150 ... 101 65 457 460 429 456
    ID          (obs) float64 1.0 2.0 3.0 4.0 ... 1.95e+06 1.95e+06 1.95e+06

#### Select eddies without any ice cover in 3R

In [9]:
dt_masked_ac_0 = dt_masked_ac.where(dt_masked_ac["mean_conc"] == 0, drop = True)

In [10]:
dt_masked_ac_0

<xarray.Dataset>
Dimensions:     (obs: 1786171)
Coordinates:
  * obs         (obs) int64 0 1 2 3 4 ... 1982336 1982337 1982338 1982339
Data variables:
    mean_conc   (obs) float64 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    npoints_ac  (obs) float64 28.0 27.0 33.0 58.0 71.0 ... 36.0 35.0 34.0 22.0
    ID          (obs) float64 1.0 2.0 3.0 4.0 ... 1.987e+06 1.987e+06 1.987e+06

In [11]:
ids_ac = dt_masked_ac_0["ID"].values
edso_ac_00 = edso_ac_0.where(edso_ac_0["ID"].isin(ids_ac), drop=True)

In [12]:
edso_ac_00

<xarray.Dataset>
Dimensions:                        (obs: 1786171, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/32)
    amplitude                      (obs) float64 0.008 0.01 ... 0.0324 0.0275
    cost_association               (obs) float32 0.2152 0.2833 ... 9.969e+36
    effective_area                 (obs) float32 5.428e+08 ... 5.388e+08
    effective_contour_height       (obs) float32 -0.048 -0.048 ... 0.188 0.196
    effective_contour_latitude     (obs, NbSample) float64 -75.94 ... -69.06
    effective_contour_longitude    (obs, NbSample) float64 203.5 203.6 ... 12.75
    ...                             ...
    track                          (obs) float64 17.0 17.0 ... 5.648e+05
    uavg_profile                   (obs, NbSample) float64 0.0719 ... 0.1291
    clt                            (obs) float32 0.9306 0.9621 ... 0.6758 0.7295
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ID                             (obs) float64 1.0 2.0 ... 1.987e+06 1.987e+06
    sst                            (obs) float32 2.876 3.377 ... 1.763 2.247
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [13]:
dt_masked_c_0 = dt_masked_c.where(dt_masked_c["mean_conc"] == 0, drop = True)

In [14]:
dt_masked_c_0

<xarray.Dataset>
Dimensions:     (obs: 1726939)
Coordinates:
  * obs         (obs) int64 0 1 2 3 4 ... 1945334 1945335 1945336 1945337
Data variables:
    mean_conc   (obs) float64 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    npoints_ac  (obs) float64 173.0 175.0 154.0 150.0 ... 460.0 429.0 456.0
    ID          (obs) float64 1.0 2.0 3.0 4.0 ... 1.95e+06 1.95e+06 1.95e+06

In [15]:
ids_c = dt_masked_c_0["ID"].values
edso_c_00 = edso_c_0.where(edso_c_0["ID"].isin(ids_c), drop=True)

In [16]:
edso_c_00

<xarray.Dataset>
Dimensions:                        (obs: 1726939, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/32)
    amplitude                      (obs) float64 -0.1169 -0.1162 ... -0.1393
    cost_association               (obs) float32 0.06921 0.1143 ... 9.969e+36
    effective_area                 (obs) float32 5.522e+09 ... 1.738e+10
    effective_contour_height       (obs) float32 0.076 0.068 ... -0.06 -0.06
    effective_contour_latitude     (obs, NbSample) float64 -70.13 ... -61.86
    effective_contour_longitude    (obs, NbSample) float64 355.8 355.5 ... 237.2
    ...                             ...
    track                          (obs) float64 5.0 5.0 ... 6.283e+05 6.283e+05
    uavg_profile                   (obs, NbSample) float64 0.2521 ... 0.1695
    clt                            (obs) float32 0.9761 0.9765 ... 0.782 0.9581
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ID                             (obs) float64 1.0 2.0 ... 1.95e+06 1.95e+06
    sst                            (obs) float32 1.564 1.696 ... 5.454 5.528
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Cyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T08:03:27Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [26]:
dt_masked_ac_1 = dt_masked_ac.where(dt_masked_ac["mean_conc"] > 0, drop = True)

In [27]:
dt_masked_ac_1

<xarray.Dataset>
Dimensions:     (obs: 333884)
Coordinates:
  * obs         (obs) int64 265 266 267 1105 ... 1986533 1986569 1986570 1986571
Data variables:
    mean_conc   (obs) float64 9.737e-06 0.009403 0.001647 ... 6.32e-05 0.0014
    npoints_ac  (obs) float64 150.0 208.0 177.0 1.988e+03 ... 366.0 370.0 503.0
    ID          (obs) float64 266.0 267.0 268.0 ... 1.987e+06 1.987e+06

In [28]:
ids_ac_1 = dt_masked_ac_1["ID"].values
edso_ac_1 = edso_ac_0.where(edso_ac_0["ID"].isin(ids_ac_1), drop=True)

In [29]:
edso_ac_1

<xarray.Dataset>
Dimensions:                        (obs: 333884, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/32)
    amplitude                      (obs) float64 0.0129 0.0151 ... 0.0265 0.0335
    cost_association               (obs) float32 0.3172 0.1749 ... 9.969e+36
    effective_area                 (obs) float32 1.828e+09 ... 5.598e+09
    effective_contour_height       (obs) float32 -0.028 -0.044 ... -0.032 -0.036
    effective_contour_latitude     (obs, NbSample) float64 -66.12 ... -67.56
    effective_contour_longitude    (obs, NbSample) float64 71.5 71.57 ... 28.5
    ...                             ...
    track                          (obs) float64 52.0 52.0 ... 5.647e+05
    uavg_profile                   (obs, NbSample) float64 0.0648 ... 0.1034
    clt                            (obs) float32 0.7949 0.6825 ... 0.9632 0.9435
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ID                             (obs) float64 266.0 267.0 ... 1.987e+06
    sst                            (obs) float32 -0.2946 -0.4016 ... 2.795 3.589
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [30]:
dt_masked_c_1 = dt_masked_c.where(dt_masked_c["mean_conc"] > 0, drop = True)

In [31]:
dt_masked_c_1

<xarray.Dataset>
Dimensions:     (obs: 360899)
Coordinates:
  * obs         (obs) int64 18 19 20 21 22 ... 1949870 1949871 1949872 1949873
Data variables:
    mean_conc   (obs) float64 0.0001083 0.00599 0.01009 ... 0.09926 0.08377
    npoints_ac  (obs) float64 151.0 202.0 127.0 129.0 ... 61.0 100.0 123.0 134.0
    ID          (obs) float64 19.0 20.0 21.0 22.0 ... 1.95e+06 1.95e+06 1.95e+06

In [32]:
ids_c_1 = dt_masked_c_1["ID"].values
edso_c_1 = edso_c_0.where(edso_c_0["ID"].isin(ids_ac_1), drop=True)

#### Export Data

In [17]:
edso_ac_00.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso_ac_3R_0.nc")

In [19]:
edso_c_00.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso_c_3R_0.nc")